# Satellite-Based Chlorophyll and NDCI Extraction for Lake Monitoring

This notebook extracts chlorophyll-a concentration estimates and Normalized Difference Chlorophyll Index (NDCI) values from multiple satellite platforms (MODIS-Terra, MODIS-Aqua, and Sentinel-2) for specified lake locations.

## Overview

- Purpose: Generate time series of chlorophyll indices from satellite imagery
- Study Areas: Detroit Lake and Upper Klamath Lake
- Satellite Sensors: MODIS (Terra and Aqua), Sentinel-2.
- Reference Satellite Sensors: Landsat 5/7/8 for reference.
- Output: CSV files with date-stamped chlorophyll/NDCI values

## Key Features

- Cloud masking using QA bands
- Water detection using NDWI (Normalized Difference Water Index)
- 500 m x 500 m spatial averaging around lake centers
- Multi-sensor harmonization for long-term monitoring

## Satellite Sensor Characteristics

| **Characteristic** | **MODIS (Terra + Aqua)** | **Sentinel-2 (2A + 2B)** | **Sentinel-3 (3A + 3B)** |
|---|---|---|---|
| **Satellites** | **Terra** (launched 1999) + **Aqua** (launched 2002) | **Sentinel-2A** (launched 2015) + **Sentinel-2B** (launched 2017) | **Sentinel-3A** (launched 2016) + **Sentinel-3B** (launched 2018) |
| **Sensor Identity** | Both carry identical MODIS instruments | Both carry identical Multi-spectral Instruments (MSI) | Both carry identical OLCI, SLSTR, SRAL |
| **Overpass Times** | Terra (VNIR): ~10:30 AM (descending) local time<br>Aqua (VNIR): ~1:30 PM (ascending) local time | Both: ~10:30 AM (descending) local solar time | Both: ~10:00 AM (descending) local solar time |
| **Temporal Resolution** | Combined: 1-2 observations/day at equator<br>Individual: Daily | Combined: 5 days at equator<br>Individual: 10 days<br>(2-3 days at mid-latitudes) | Combined: <2 days at equator<br>Individual: ~2 days<br>(Better than daily at mid-latitudes) |
| **Orbital Configuration** | Polar sun-synchronous orbits<br>Different crossing times provide morning/afternoon views | Polar sun-synchronous orbits<br>Phased 180° apart | Polar sun-synchronous orbits<br>Phased 140° apart |
| **Spatial Resolution** | 250m (bands 1-2)<br>500m (bands 3-7)<br>1000m (bands 8-36) | 10m (visible, NIR)<br>20m (red-edge, SWIR)<br>60m (atmospheric bands) | OLCI: 300m (21 bands)<br>SLSTR: 500m (VIS/NIR), 1km (TIR) |
| **Spectral Bands** | 36 bands (0.4-14.4 μm) | 13 bands (0.44-2.19 μm) | OLCI: 21 bands (0.4-1.02 μm)<br>SLSTR: 11 bands (0.55-12 μm) |
| **Key Strengths** | • Complementary morning/afternoon observations<br>• Captures diurnal variations<br>• 25+ year data continuity<br>• Established ocean color algorithms | • Twin satellites reduce revisit time<br>• Consistent 10:30 AM observations<br>• High spatial detail (10m)<br>• Three red-edge bands<br>• Optimized for land/coastal monitoring | • Better than daily coverage<br>• Dedicated ocean color sensor (OLCI)<br>• Improved spatial resolution vs MODIS<br>• Multiple instruments per satellite<br>• Optimized for water applications |
| **Limitations** | • Coarse spatial resolution<br>• Morning (Terra) may have more clouds<br>• Afternoon (Aqua) may miss morning blooms | • Limited diurnal information<br>• Both pass at same local time<br>• Shorter historical record | • Moderate spatial resolution<br>• Limited historical record<br>• Both satellites at similar overpass time |
| **Chlorophyll Applications** | • Large water bodies<br>• Open ocean monitoring<br>• Diurnal bloom dynamics<br>• Long-term trend analysis | • Small/medium lakes<br>• Coastal zones<br>• Spatial heterogeneity<br>• Near-shore processes | • Medium/large lakes<br>• Coastal/ocean monitoring<br>• Daily bloom tracking<br>• Operational monitoring |
| **Chlorophyll-Relevant Bands** | Blue: B9 (443 nm), B10 (488 nm) - 1km<br>Green: B11 (531 nm), B12 (551 nm) - 1km<br>Red: B13 (667 nm), B14 (678 nm) - 1km<br>NIR: B15 (748 nm), B16 (869 nm) - 1km<br>Note: B1 (645 nm) & B2 (859 nm) at 250m sometimes used | Blue: B2 (490 nm) - 10m<br>Green: B3 (560 nm) - 10m<br>Red: B4 (665 nm) - 10m<br>Red-edge: B5 (705 nm), B6 (740 nm), B7 (783 nm) - 20m<br>NIR: B8 (842 nm) - 10m | OLCI Blue: Oa03 (442.5 nm), Oa04 (490 nm) - 300m<br>OLCI Green: Oa05 (510 nm), Oa06 (560 nm) - 300m<br>OLCI Red: Oa08 (665 nm), Oa09 (674 nm) - 300m<br>OLCI Fluorescence: Oa10 (681 nm) - 300m<br>OLCI Red-edge: Oa11 (709 nm), Oa12 (754 nm) - 300m |
| **Chlorophyll-Relevant Bands** | Blue: B3 (469 nm) - 500m<br>B9 (443 nm), B10 (488 nm) - 1km<br>Green: B4 (555 nm) - 500m<br>B11 (531 nm), B12 (551 nm) - 1km<br>Red: B1 (645 nm) - 250m<br>B13 (667 nm), B14 (678 nm) - 1km<br>NIR: B2 (859 nm) - 250m<br>B15 (748 nm), B16 (869 nm) - 1km<br>Red-edge: Not available | Blue: B2 (490 nm) - 10m<br>Green: B3 (560 nm) - 10m<br>Red: B4 (665 nm) - 10m<br>Red-edge: B5 (705 nm), B6 (740 nm), B7 (783 nm) - 20m<br>NIR: B8 (842 nm) - 10m | OLCI Blue: Oa03 (442.5 nm), Oa04 (490 nm) - 300m<br>OLCI Green: Oa05 (510 nm), Oa06 (560 nm) - 300m<br>OLCI Red: Oa08 (665 nm), Oa09 (674 nm) - 300m<br>OLCI Fluorescence: Oa10 (681 nm) - 300m<br>OLCI Red-edge: Oa11 (709 nm), Oa12 (754 nm) - 300m |



| **Characteristic** | **MODIS (Terra + Aqua)** | **Sentinel-2 (2A + 2B)** | **Sentinel-3 (3A + 3B)** |
|---|---|---|---|
| **Common Chlorophyll Indices** | • OC3M: Uses B9, B10, B12<br>• NDCI: (B14-B13)/(B14+B13)<br>• FLH: Fluorescence Line Height using B13, B14, B15<br>• Two-band ratio: B10/B12 or B9/B12 | • NDCI: (B5-B4)/(B5+B4)<br>• 2BDA: B4/B3<br>• 3BDA: (1/B4 - 1/B5) × B7<br>• MCI: B5 - B4 - (B6-B4) × [(705-665)/(740-665)]<br>• Multiple red-edge indices available | • OC4Me: Uses Oa03, Oa04, Oa05, Oa06<br>• MCI: Maximum Chlorophyll Index using Oa08, Oa09, Oa10<br>• FLH: Uses Oa08, Oa10, Oa11<br>• NDCI: (Oa11-Oa08)/(Oa11+Oa08)<br>• Dedicated algae products (Level-2) |
| **Chlorophyll-Relevant Bands** | **250m**: B1 (645 nm), B2 (859 nm)<br>**500m**: B3 (469 nm), B4 (555 nm), B5 (1240 nm), B6 (1640 nm), B7 (2130 nm)<br>**1km**: B8-B16 including B9 (443 nm), B10 (488 nm), B11 (531 nm), B12 (551 nm), B13 (667 nm), B14 (678 nm) | **10m**: B2 (490 nm), B3 (560 nm), B4 (665 nm), B8 (842 nm)<br>**20m**: B5 (705 nm), B6 (740 nm), B7 (783 nm)<br>**60m**: B1 (443 nm), B9 (945 nm) | **OLCI (300m)**: Oa03 (442.5 nm), Oa04 (490 nm), Oa05 (510 nm), Oa06 (560 nm), Oa08 (665 nm), Oa09 (674 nm), Oa10 (681 nm), Oa11 (709 nm), Oa12 (754 nm) |
| **Your Selected Bands/Indices** | **YOUR METHOD**: Green-to-red ratio<br>**Bands**: B4 (555 nm) / B1 (645 nm)<br>**Resolution**: 500m / 250m<br>**Algorithm**: log₁₀(Chl) = 1.70 × log₁₀(R₅₅₅/R₆₄₅) + 1.54 | **YOUR METHOD**: NDCI<br>**Bands**: (B5 - B4) / (B5 + B4)<br>**Wavelengths**: (705 - 665) / (705 + 665)<br>**Resolution**: 20m / 10m | **RECOMMENDED**: <br>• OC4Me algorithm<br>• MCI using Oa08, Oa09, Oa10<br>• NDCI: (Oa11-Oa08)/(Oa11+Oa08)<br>• Level-2 chlorophyll product |

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

## MODIS Processing (500m resolution)

### Overview
MODIS (Moderate Resolution Imaging Spectroradiometer) provides daily global coverage in the visible bands at 250, 500, and 1,000 m resolution from two satellites: Terra (morning overpass) and Aqua (afternoon overpass). This section derives chlorophyll-a concentrations using an empirical green-to-red ratio algorithm using the 500 m bands.

### Chlorophyll Algorithm:

The algorithm uses the ratio of green (555 nm) to red (645 nm) reflectance:

$$
log_{10}(Chl-a) = a × log_{10}(R_{rs,555}/R_{rs,645}) + b
$$

Where:
- $R_{rs}$ is the remote sensing reflectance ($sr/π$)
- a and b are coefficients calibrated for inland waters
  - a = 1.70
  - b = 1.54

### Key Features:

- Daily coverage: Maximizes temporal resolution for trend analysis
- Dual satellites: Terra and Aqua provide two observations per day. In the visible and near-infrared bands, their nominal overpass times are:
  - Terra: 10:30 AM local standard time
  - Aqua: 1:30 PM local standard time
- Long record: Continuous data since 2000 (Terra) and 2002 (Aqua)
- Trade-off: Lower spatial resolution (500 m) but higher temporal frequency


In [ ]:
# ============================================================================
# MODIS Chlorophyll-a Extraction
# ============================================================================
"""
Generate daily chlorophyll-a time series from MODIS Terra and Aqua satellites.
Each lake receives two separate CSV files (one per satellite) with Chl-a estimates.
"""

# ------------ USER CONFIGURATION --------------------------------------------
# Define lake locations and output filenames
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         terra_export='Detroit_MODIS_Terra_500m_Chl_singlePixel',
         aqua_export='Detroit_MODIS_Aqua_500m_Chl_singlePixel'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         terra_export='Klamath_MODIS_Terra_500m_Chl_singlePixel',
         aqua_export='Klamath_MODIS_Aqua_500m_Chl_singlePixel')
    # Add more lakes here if desired
]

# ------------ GLOBAL SETTINGS -----------------------------------------------
# Temporal range for analysis
start_date = '2011-01-01'
end_date = '2025-12-31'

# Chlorophyll algorithm coefficients (green:red ratio method)
# Based on empirical calibration for inland waters
offset = 1.54  # Y-intercept (use 0.96 for UKL low-end adjust if needed)
slope = 1.70   # Slope coefficient

# ------------ PREPROCESSING FUNCTIONS ---------------------------------------
def mask_light_cloud(img):
    """
    Apply light cloud mask using state_1km QA band.
    
    QA bit flags:
    - Bit 10: Cloud state
    - Bit 12: Snow/ice flag
    
    This is a lighter mask than strict cloud detection to preserve more data.
    """
    qa = img.select('state_1km')
    cloud = qa.bitwiseAnd(1 << 10).neq(0)
    snow = qa.bitwiseAnd(1 << 12).neq(0)
    return img.updateMask(cloud.Not()).updateMask(snow.Not())

def add_chlorophyll(img):
    """
    Calculate chlorophyll-a concentration using green-to-red ratio algorithm.
    
    Algorithm:
    1. Convert surface reflectance to remote sensing reflectance (Rrs = sr/π)
    2. Calculate log10 ratio of green (555nm) to red (645nm)
    3. Apply linear regression: log10(Chl) = slope × log10(ratio) + offset
    4. Convert from log10 to linear scale
    
    Bands used:
    - Band 4 (sur_refl_b04): Green (545-565 nm)
    - Band 1 (sur_refl_b01): Red (620-670 nm)
    
    Returns:
        Image with added 'chlor_a' band in µg/L
    """
    # Convert to surface reflectance (scale factor = 1e-4)
    sr555 = img.select('sur_refl_b04').multiply(1e-4)  # Green (555 nm)
    sr645 = img.select('sur_refl_b01').multiply(1e-4)  # Red (645 nm)
    
    # Convert to remote sensing reflectance (divide by π)
    rrs555 = sr555.divide(ee.Number(math.pi))
    rrs645 = sr645.divide(ee.Number(math.pi))
    
    # Calculate log10 of green/red ratio
    log_ratio = rrs555.divide(rrs645).log10()
    
    # Apply empirical algorithm: log10(Chl) = slope × log_ratio + offset
    log10_chl = log_ratio.multiply(slope).add(offset)
    
    # Convert from log10 to linear scale (µg/L)
    chl = ee.Image(10).pow(log10_chl).rename('chlor_a')
    
    return img.addBands(chl)

def build_series(col_id, sensor_tag, pt):
    """
    Build chlorophyll time series for a specific MODIS collection and location.
    
    Args:
        col_id: GEE collection ID (MOD09GA for Terra, MYD09GA for Aqua)
        sensor_tag: Label for the sensor ('Terra' or 'Aqua')
        pt: Point geometry for sampling location
    
    Returns:
        Feature collection with date, chlorophyll value, and sensor tag
    """
    # Process MODIS collection
    collection = (ee.ImageCollection(col_id)
                  .filterDate(start_date, end_date)
                  .filterBounds(pt)
                  .map(mask_light_cloud)
                  .map(add_chlorophyll)
                  .select('chlor_a'))
    
    def img_to_feature(img):
        """
        Extract chlorophyll value at point location.
        
        Samples single 500m pixel at lake center point.
        Returns None if no valid data available.
        """
        # Sample single pixel at point location
        fc = img.sample(region=pt,
                        scale=500,      # MODIS pixel size
                        numPixels=1,    # Single pixel
                        geometries=False)
        
        # Return feature only if valid data exists
        return ee.Algorithms.If(
            fc.size().gt(0),
            ee.Feature(None, {
                'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
                'chl': fc.first().get('chlor_a'),
                'sensor': sensor_tag
            }),
            None)
    
    return collection.map(img_to_feature, dropNulls=True)

# ------------ PROCESS EACH LAKE ---------------------------------------------
for lake in lakes:
    # Define point geometry for lake center
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    
    # Build time series for both Terra and Aqua
    terra_series = build_series('MODIS/061/MOD09GA', 'Terra', center)
    aqua_series = build_series('MODIS/061/MYD09GA', 'Aqua', center)
    
    # Print data availability
    print(lake['name'], 'Terra rows =',
          terra_series.aggregate_count('chl').getInfo())
    print(lake['name'], 'Aqua rows =',
          aqua_series.aggregate_count('chl').getInfo())
    
    # Export Terra data to CSV
    rows = terra_series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['terra_export'] + '.csv', index=False)
    
    # Export Aqua data to CSV
    rows = aqua_series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['aqua_export'] + '.csv', index=False)
    
    # Alternative: Export to Google Drive (server-side)
    # ee.batch.Export.table.toDrive(
    #     collection=terra_series,
    #     description=lake['terra_export'],
    #     fileFormat='CSV'
    # ).start()
    #
    # ee.batch.Export.table.toDrive(
    #     collection=aqua_series,
    #     description=lake['aqua_export'],
    #     fileFormat='CSV'
    # ).start()

Detroit Terra rows = 1841
Detroit Aqua rows = 1867
UpperKlamath Terra rows = 2682
UpperKlamath Aqua rows = 2587


## Sentinel-2 Processing (10m resolution)

### Overview
Sentinel-2 provides high spatial (10-20m) and temporal (5-day revisit) resolution multispectral imagery since 2015. The constellation consists of two satellites (2A and 2B) offering 13 spectral bands optimized for vegetation and water monitoring.

### Key Processing Steps:
1. **Cloud Masking**: Uses QA60 band to identify and remove clouds and cirrus
2. **Water Detection**: NDWI thresholding to isolate water bodies
3. **NDCI Calculation**: Uses red (665 nm) and red-edge (705 nm) bands
4. **Quality Control**: Removes edge artifacts and very dark pixels

### Advantages over Landsat:
- Higher spatial resolution (10m vs 30m)
- Red-edge band specifically designed for chlorophyll detection
- More frequent revisits (5 days vs 16 days)

In [ ]:
# ============================================================================
# SENTINEL-2 NDCI Extraction
# ============================================================================

# ------------ USER CONFIGURATION --------------------------------------------
# Define lake locations and output filenames
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_S2_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_S2_NDCI_500m')
    # Add more lakes here if needed
]

# Temporal range (Sentinel-2 available from July 2015)
start_date, end_date = '2015-07-01', '2025-12-31'

# Spatial buffer for 500m x 500m ROI
half_size_m = 250  # metres

# ------------ SATELLITE COLLECTION ------------------------------------------
# Sentinel-2 Level-2A Surface Reflectance (harmonized collection)
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# ------------ PREPROCESSING FUNCTIONS ---------------------------------------
def mask_s2(img):
    """
    Apply cloud and cirrus mask using QA60 band.
    
    QA60 bit flags:
    - Bit 10: Opaque clouds
    - Bit 11: Cirrus clouds
    
    Also removes edge artifacts where B8A (NIR narrow) = 0
    """
    qa = img.select('QA60')
    cloud = qa.bitwiseAnd(1 << 10).neq(0)   # Opaque cloud flag
    cirrus = qa.bitwiseAnd(1 << 11).neq(0)  # Cirrus cloud flag
    mask = cloud.Or(cirrus).Not()           # Clear pixels
    
    # Remove edge stripes where B8A = 0
    return img.updateMask(mask)\
              .updateMask(img.select('B8A').gt(0))

def add_water_mask_s2(img):
    """
    Apply water mask using NDWI.
    
    NDWI = (Green - NIR) / (Green + NIR)
    Uses B3 (Green, 560nm) and B8 (NIR, 842nm)
    """
    ndwi = img.normalizedDifference(['B3', 'B8'])
    return img.updateMask(ndwi.gt(0))

def add_ndci(img):
    """
    Calculate NDCI using red and red-edge bands.
    
    NDCI = (Red-edge - Red) / (Red-edge + Red)
    - B4: Red band (665 nm) - sensitive to chlorophyll absorption
    - B5: Red-edge band (705 nm) - sensitive to vegetation/algae
    
    Scale factor for Sentinel-2 L2A = 1e-4
    """
    # Convert to surface reflectance
    sr = img.select(['B4', 'B5']).multiply(1e-4)
    red = sr.select('B4')   # 665 nm
    edge = sr.select('B5')  # 705 nm (red-edge)
    
    # Calculate NDCI
    ndci = edge.subtract(red)\
               .divide(edge.add(red))\
               .rename('NDCI')
    return img.addBands(ndci)

def preprocess_s2(img):
    """
    Complete preprocessing pipeline for Sentinel-2 imagery.
    
    Steps:
    1. Cloud/cirrus masking
    2. Water detection
    3. Remove very dark pixels
    """
    img = mask_s2(img)            # QA60 cloud/cirrus mask
    img = add_water_mask_s2(img)  # NDWI > 0 for water
    
    # Remove very dark red-edge pixels (shadows, poor quality)
    edge = img.select('B5').multiply(1e-4)
    img = img.updateMask(edge.gt(0.002))
    
    return img

def img_to_feature(img, roi, tag):
    """
    Convert image to feature with mean NDCI over ROI.
    
    Args:
        img: Sentinel-2 image with NDCI band
        roi: Region of interest geometry
        tag: Sensor identifier tag
    
    Returns:
        Feature with date, NDCI value, and sensor tag
    """
    # Calculate mean NDCI within ROI
    mean = img.select('NDCI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,  # Sentinel-2 pixel size (10m for visible/NIR bands)
        maxPixels=1e9
    ).get('NDCI')
    
    # Return feature only if valid NDCI exists
    return ee.Algorithms.If(
        mean,
        ee.Feature(None, {
            'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'ndci': mean,
            'sensor': tag
        }),
        None)

# ------------ PROCESS EACH LAKE ---------------------------------------------
for lake in lakes:
    # Define ROI as 500m x 500m box centered on lake
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(half_size_m).bounds()
    
    # Process Sentinel-2 collection
    series = (s2.filterDate(start_date, end_date)
               .filterBounds(roi)
               .map(preprocess_s2)
               .map(add_ndci)
               .map(lambda img: img_to_feature(img, roi, lake['name'] + '_S2'),
                    dropNulls=True))
    
    # Print scene count
    print(lake['name'],
          'valid Sentinel-2 scenes =',
          series.aggregate_count('ndci').getInfo())
    
    # Export to CSV (client-side processing)
    rows = series.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df.to_csv(lake['export_id'] + '.csv', index=False)
    
    # Alternative: Export to Google Drive (server-side)
    # ee.batch.Export.table.toDrive(
    #     collection=series,
    #     description=lake['export_id'],
    #     fileFormat='CSV'
    # ).start()

Detroit valid Sentinel-2 scenes = 286
UpperKlamath valid Sentinel-2 scenes = 1556
